# Facial Recognition


## Import


In [ ]:
import os
import time
import csv
from collections.abc import Callable
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, TypeAlias, cast

import cv2 as cv
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
from deepface import DeepFace
from deepface.modules.exceptions import FaceNotDetected


## Settings


In [ ]:
PRERECORDING_PATH = Path('data/on_site.mp4')

if 'SSH_CLIENT' in os.environ:
	print('Remote session detected. Using video file.')
	VIDEO_SOURCE = PRERECORDING_PATH
else:
	print('Local session detected. Using live webcam.')
	VIDEO_SOURCE = 0

RECORDING_FPS=30.0
OUTPUT_PATH = Path('data/output.avi')

RECOGNITION_MODEL = 'Facenet'
DISTANCE_METRIC = 'euclidean_l2'
DETECTOR_BACKEND = 'ssd'

DB_PATH = Path('data/faces_db')
TOP_K = 1
BOX_COLOUR = (0, 0, 255)
"""BGR format."""
TEXT_COLOUR = (50, 50, 230)
"""BGR format."""

PATIENCE = 10
"""How many frames before swapping state."""

CSV_PATH = Path('log.csv')


## Generic CV Function


In [ ]:
# Over-engineered to all hell
class VideoProcessor:

	@dataclass
	class DisplayConfig:
		pass

	@dataclass
	class DisplayOption_Headless(DisplayConfig):
		pass

	@dataclass
	class DisplayOption_Jupyter(DisplayConfig):
		image_widget: widgets.Image

	@dataclass
	class DisplayOption_OpenCV(DisplayConfig):
		pass

	#DisplayConfig: TypeAlias = DisplayOption_Headless | DisplayOption_Jupyter | DisplayOption_OpenCV

	FrameCallbackType: TypeAlias = Callable[[cv.typing.MatLike], cv.typing.MatLike | None]

	_END_LOOP = False
	_GO_AGAIN = True

	@staticmethod
	def _display_video(
		display_option: DisplayConfig,
		frame: cv.typing.MatLike
	) -> bool:
		"""Will attempt to display an output based on the display option."""

		match display_option:
			case VideoProcessor.DisplayOption_Jupyter(image_widget=image_widget):
				ret: bool
				buffer: np.ndarray[Any, np.dtype[np.uint8]]
				ret, buffer = cv.imencode(ext='.jpg', img=frame)

				if not ret:
					print("Can't encode frame as image. Exiting ...")
					return VideoProcessor._END_LOOP

				image_widget.value = buffer.tobytes()

			case VideoProcessor.DisplayOption_OpenCV:
				print('Frame should be displayed')
				cv.imshow(winname='frame', mat=frame)

		return VideoProcessor._GO_AGAIN

	@staticmethod
	def _frame_loop(
		vid_cap: cv.VideoCapture,
		callback: FrameCallbackType,
		display_option: DisplayConfig,
		frametime: float
	) -> bool:
		"""This function is called continuously until either the video ends, or is interrupted."""

		start_time: float = time.time()

		ret: bool
		frame: cv.typing.MatLike
		ret, frame = vid_cap.read()

		if not ret:
			print("Can't receive frame (stream end?). Exiting ...")
			return VideoProcessor._END_LOOP

		frame_result: cv.typing.MatLike | None = callback(frame)
		if frame_result is None: frame_result = frame

		if not VideoProcessor._display_video(
			display_option=display_option,
			frame=frame_result
		): return VideoProcessor._END_LOOP

		match display_option:
			case VideoProcessor.DisplayOption_OpenCV:
				if cv.waitKey(delay=int(1/frametime)) == ord('q'):
					return VideoProcessor._END_LOOP

			case _: # VideoProcessor.DisplayOption_Jupyter:
				# Enforce the framerate pacing
				elapsed_time: float = time.time() - start_time
				time_to_wait: float = frametime - elapsed_time
				if time_to_wait > 0: time.sleep(time_to_wait)

		return VideoProcessor._GO_AGAIN

	@staticmethod
	def process_video(
		capture_location: int | str | Path,
		/,
		callback: FrameCallbackType = lambda x: x,
		*,
		display_config: DisplayConfig | None = None,
		frametime: float = 1.0 / 30.0
	) -> None:
		"""Read from a video input and apply the callback to it.

		Args:
			capture_location (int | str | Path): What the video source is.
				- For a webcam input, use an int.
				- For a video file, use a str or Path.
			callback (FrameCallbackType: Callable[[cv.typing.MatLike], cv.typing.MatLike | None]): Function that will be called with each frame.
			frametime (float): In seconds, how long between each frame. Use 1 / fps if you want to pass in a framerate.
			display_type (DisplayType): Whether to output the frames, and if so, how should it be shown. Options:
				- Jupyter Notebook.
				- Qt window via OpenCV.

		Returns:
			None:
		"""

		# Null sentinel
		if display_config is None:
			display_config = VideoProcessor.DisplayOption_Jupyter(
				# JPEG is faster than PNG
				image_widget=widgets.Image(format='jpeg')
			)

		vid_cap = cv.VideoCapture(capture_location)

		if not vid_cap.isOpened():
			print('Cannot open video source.')
			return

		if isinstance(display_config, VideoProcessor.DisplayOption_Jupyter):
			display(display_config.image_widget)

		try:
			while VideoProcessor._frame_loop(
				vid_cap=vid_cap,
				callback=callback,
				display_option=display_config,
				frametime=frametime
			): pass

		except KeyboardInterrupt:
			print('Video stream interrupted.')

		finally:
			vid_cap.release()


## Prebuild Model


In [ ]:
_ = DeepFace.build_model(model_name=RECOGNITION_MODEL)


## Get Faces In Frame


In [ ]:
def get_faces(frame: cv.typing.MatLike, /) -> list[pd.DataFrame] | None:
	try:
		dfs: list[pd.DataFrame] | list[list[dict[str, Any]]] = DeepFace.find(
			img_path=frame,
			db_path=str(object=DB_PATH),
			model_name=RECOGNITION_MODEL,
			distance_metric=DISTANCE_METRIC,
			detector_backend=DETECTOR_BACKEND,
			k=TOP_K,
			normalization=RECOGNITION_MODEL,
			silent=True
		)
	except FaceNotDetected:
		return None

	# Genuinely fucked way to detect batch return type
	#match dfs:
	#	case list() as inner:
	#		match inner:
	#			case list() as batched:
	#				raise Exception('Doing batched for some reason')

	#return dfs

	#match dfs:
	#	case [list(), *_]:
	#		# Matches a sequence where the first element is a list
	#		raise Exception('Doing batched for some reason')
	#	case [pd.DataFrame(), *_]:
	#		# Matches a sequence where the first element is a DataFrame
	#		return dfs

	if dfs and isinstance(dfs[0], list):
		raise Exception('Doing batched for some reason')

	if not len(dfs):
		return None

	return cast(list[pd.DataFrame], dfs)


## Draw Onto Frame


In [ ]:
def extract_name(identity: str, /) -> str:
	id_path = Path(identity)
	return ''.join(id_path.parts[-2:-1])


In [ ]:
def draw_onto_frame(
	frame: cv.typing.MatLike,
	face: pd.Series,
	name: str
) -> cv.typing.MatLike:
	left: int = face['source_x']
	top: int = face['source_y']
	right: int = face['source_w']
	bottom: int = face['source_h']

	# Rectangle
	frame = cv.rectangle(
		img=frame,
		rec=(left, top, right, bottom),
		color=BOX_COLOUR
	)

	# Text
	cv.putText(
		img=frame,
		text=name,
		org=(left, top),
		fontFace=cv.FONT_HERSHEY_SIMPLEX,
		fontScale=0.8,
		color=TEXT_COLOUR,
		thickness=2,
		lineType=cv.LINE_AA
	)

	return frame


## Output To CSV


In [ ]:
# Clear the file
with open(file=CSV_PATH, mode='w') as csvfile:
	pass

def write_to_csv(name: str, event: bool) -> None:
	event_text: str = 'ENTER' if event else 'EXIT'
	current_time = str(object=datetime.now().strftime(format='%Y-%m-%d %H:%M:%S'))
	with open(file=CSV_PATH, mode='a') as csvfile:
		writer: csv.DictWriter[str] = csv.DictWriter(f=csvfile, fieldnames=['Name', 'Event', 'Time'])
		writer.writerow(rowdict={'Name': name, 'Event': event_text, 'Time': current_time})


In [ ]:
@dataclass
class Person:
	present: bool = False
	timer: int = 0

# Name and info
log: dict[str, Person] = {}

def update_log(present: list[str]) -> None:
	# Update known people
	for name, person in log.items():
		is_present: bool = name in present
		# Check state change
		if person.present != is_present:
			# Check current timer
			if person.timer == PATIENCE:
				person.timer = 0
				person.present = is_present
				write_to_csv(name=name, event=is_present)
			else:
				# Update timer
				person.timer += 1

	# Check for first appearance
	for name in set(present) - log.keys():
		if not name in log:
			log[name] = Person(present=True)
			write_to_csv(name=name, event=True)


## Per-frame Callback


In [ ]:
def face_detection(frame: cv.typing.MatLike) -> cv.typing.MatLike | None:
	detected_this_frame: list[str] = []
	detected_faces: list[pd.DataFrame] | None = get_faces(frame)

	if detected_faces:
		# Only one for loop needed this way.
		for detected_face in detected_faces:
			# For some reason, you can get empty dataframes
			if detected_face.empty:
				continue

			matched_face: pd.Series[Any] = detected_face.iloc[0]

			name: str = extract_name(matched_face['identity'])

			detected_this_frame.append(name)

			frame = draw_onto_frame(
				frame=frame,
				face=matched_face,
				name=name
			)

	update_log(detected_this_frame)

	return frame


## Run


In [ ]:
VideoProcessor.process_video(
	VIDEO_SOURCE,
	callback=face_detection,
	frametime=1./10
)


## Output Display


### Display Video Source


In [ ]:
# Will display webcam when run locally, or video file when run remotely
VideoProcessor.process_video(
	VIDEO_SOURCE,
	callback=lambda frame: cv.cvtColor(frame, cv.COLOR_RGB2GRAY)
)


### Display Prerecorded Video


In [ ]:
# Will always display the video file
VideoProcessor.process_video(
	PRERECORDING_PATH,
	callback=lambda frame: cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
)


### Record Webcam


In [ ]:
# Define the codec and create VideoWriter object
fourcc = cv.VideoWriter_fourcc(*'XVID') # type: ignore
out = cv.VideoWriter(
	filename=OUTPUT_PATH,
	fourcc=fourcc,
	fps=RECORDING_FPS,
	frameSize=(640,  480)
)

VideoProcessor.process_video(
	0,
	callback=lambda frame: out.write(frame),
	frametime=1.0 / RECORDING_FPS
)

out.release()
